#### Q2 — VALIDATION ENGINE (Flag violating rows)  

# Business Rules (Summary)

| Rule | Business Rule                              | Applies To (Files)               | Columns Checked                            |
|------|--------------------------------------------|----------------------------------|--------------------------------------------|
| R1   | No duplicate primary keys                  | sales, orders                    | transaction_id, order_id                   |
| R2   | Date must be valid & in YYYY-MM-DD format  | sales, orders, inventory         | date, snapshot_date                        |
| R3   | Warehouse ID must exist in the master list | sales, orders, inventory, warehouses | warehouse_id                            |
| R4   | SKU code must exist in the master list     | sales, orders, inventory         | sku                                        |
| R5   | Quantity values cannot be negative         | sales, orders, inventory         | quantity, revenue, ordered_quantity, on_hand_qty |
| R6   | Order status must be one of the allowed set| orders                          | order_status                               |

In [1]:
import pandas as pd
from pathlib import Path
import logging

In [2]:
logger = logging.getLogger(__name__)

In [3]:
def parse_iso(val):  
    try:  
        dt = pd.to_datetime(val)
        return dt
    except (ValueError, TypeError):
        return None

In [4]:
def flag_rules(df, name, id_col=None, date_col=None, sku_col=None,
               wh_col=None, qty_cols=None, status_col=None,
               warehouses: Optional[pd.DataFrame] = None,
               master_sku: Optional[Set[str]] = None,
               allowed_status: Optional[Set[str]] = None
              ) -> pd.DataFrame:
    
    valid_wh = set(warehouses.iloc[:, 0]) if warehouses is not None else set()
    master_sku = {f"SKU-{i:04d}" for i in range(1, 41)}
    records = []

    def get_id(ix):
        return df.at[ix, id_col] if (id_col and id_col in df.columns) else None

    if id_col and id_col in df.columns:
        dup_mask = df.duplicated(subset=[id_col], keep="first")
        for ix in df.index[dup_mask]:
            records.append({
                "file": name, "row_index": ix, "id_value": get_id(ix),
                "rule_id": "R1", "offending_value": get_id(ix),
                "description": "Duplicate primary key",
            })

    if date_col and date_col in df.columns:
        for ix, v in df[date_col].items():
            if pd.isna(v):
                continue
            try:
                pd.to_datetime(v, format="%Y-%m-%d", errors="raise")
            except (ValueError, TypeError):
                records.append({
                    "file": name, "row_index": ix, "id_value": get_id(ix),
                    "rule_id": "R2", "offending_value": v,
                    "description": "Invalid date format",
                })

    if wh_col and wh_col in df.columns:
        bad_wh = df[~df[wh_col].isin(valid_wh)].index
        for ix in bad_wh:
            records.append({
                "file": name, "row_index": ix, "id_value": get_id(ix),
                "rule_id": "R3", "offending_value": df.at[ix, wh_col],
                "description": "Unknown warehouse_id",
            })

    if sku_col and sku_col in df.columns:
        bad_sk = df[~df[sku_col].isin(master_sku)].index
        for ix in bad_sk:
            records.append({
                "file": name, "row_index": ix, "id_value": get_id(ix),
                "rule_id": "R4", "offending_value": df.at[ix, sku_col],
                "description": "SKU not in master list",
            })

    if qty_cols:
        for c in qty_cols:
            if c not in df.columns:
                continue
            numeric = pd.to_numeric(df[c], errors="coerce")
            for ix in numeric[numeric < 0].index:
                records.append({
                    "file": name,
                    "row_index": ix,
                    "id_value": get_id(ix),
                    "rule_id": "R5",
                    "offending_value": df.at[ix, c],
                    "description": f"Negative value in '{c}'",
                })

    if status_col and status_col in df.columns:
        bad_st = df[~df[status_col].astype(str).str.strip()
                     .isin(allowed_status)].index
        for ix in bad_st:
            records.append({
                "file": name,
                "row_index": ix,
                "id_value": get_id(ix),
                "rule_id": "R6",
                "offending_value": df.at[ix, status_col],
                "description": "Non-standard status",
            })

    result = pd.DataFrame(records)
    if result.empty:
        result = pd.DataFrame(columns=[
            "file", "row_index", "id_value",
            "rule_id", "offending_value", "description",
        ])
    return result

In [5]:
def validate_all_files(
    raw_data: Dict[str, pd.DataFrame],
    warehouses: pd.DataFrame,
    master_sku: Optional[Set[str]] = None,
    allowed_status: Optional[Set[str]] = None
) -> Dict[str, pd.DataFrame]:
    """
    Validate all raw data files against business rules
    
    Args:
        raw_data: Dictionary with file names as keys and dataframes as values
        warehouses: Warehouse master dataframe
        master_sku: Set of valid SKU codes (auto-generated if None)
        allowed_status: Set of allowed order statuses
    
    Returns:
        Dictionary with file names as keys and violation dataframes as values
    """
    logger.info("=" * 60)
    logger.info("VALIDATING ALL FILES")
    logger.info("=" * 60)
    
    # Set default master SKU if not provided
    if master_sku is None:
        # Generate SKU list from SKU-0001 to SKU-0040
        master_sku = {f"SKU-{i:04d}" for i in range(1, 41)}
    
    # Set default allowed statuses if not provided
    if allowed_status is None:
        allowed_status = {"Pending", "Shipped", "Delivered", "Cancelled"}
    
    violations = {}
    summary_data = []
    
    # Define validation configuration for each file
    file_configs = {
        'sales': {
            'id_col': 'transaction_id',
            'date_col': 'date',
            'sku_col': 'sku',
            'wh_col': 'warehouse_id',
            'qty_cols': ['quantity', 'revenue'],
            'status_col': None
        },
        'orders': {
            'id_col': 'order_id',
            'date_col': 'date',
            'sku_col': 'sku',
            'wh_col': 'warehouse_id',
            'qty_cols': ['ordered_quantity'],
            'status_col': 'order_status'
        },
        'inventory': {
            'id_col': None,
            'date_col': 'snapshot_date',
            'sku_col': 'sku',
            'wh_col': 'warehouse_id',
            'qty_cols': ['on_hand_qty'],
            'status_col': None
        }
    }
    
    # Validate each file
    for file_name, df in raw_data.items():
        if file_name not in file_configs:
            logger.warning(f"No validation config for {file_name}, skipping...")
            continue
            
        logger.info(f"\nValidating {file_name}...")
        config = file_configs[file_name]
        
        # Run validation
        viol_df = flag_rules(
            df=df,
            name=file_name,
            id_col=config['id_col'],
            date_col=config['date_col'],
            sku_col=config['sku_col'],
            wh_col=config['wh_col'],
            qty_cols=config['qty_cols'],
            status_col=config['status_col'],
            warehouses=warehouses,
            master_sku=master_sku,
            allowed_status=allowed_status
        )
        
        violations[file_name] = viol_df
        
        # Log violations summary
        if not viol_df.empty:
            logger.info(f"  Found {len(viol_df)} violations:")
            for rule, count in viol_df['rule_id'].value_counts().items():
                logger.info(f"    Rule {rule}: {count} violations")
                
                # Show sample of offending values
                sample_values = viol_df[viol_df['rule_id'] == rule]['offending_value'].head(3)
                logger.info(f"      Sample values: {sample_values.tolist()}")
        else:
            logger.info(f"  No violations found!")
        
        summary_data.append({
            'file': file_name,
            'total_rows': len(df),
            'violations': len(viol_df),
            'clean_rows': len(df) - len(viol_df)
        })
    
    # Log summary
    logger.info("\n" + "=" * 60)
    logger.info("VALIDATION SUMMARY")
    logger.info("=" * 60)
    summary_df = pd.DataFrame(summary_data)
    logger.info("\n" + summary_df.to_string(index=False))
    
    return violations

In [6]:
def generate_validation_report(
    violations: Dict[str, pd.DataFrame],
    output_path: Optional[Path] = None
) -> pd.DataFrame:
    """
    Generate a comprehensive validation report
    
    Args:
        violations: Dictionary of violation dataframes
        output_path: Path to save the report (optional)
    
    Returns:
        Aggregated violation summary DataFrame
    """
    # Combine all violations
    all_violations = pd.concat(
        [df for df in violations.values() if not df.empty],
        ignore_index=True
    )
    
    if all_violations.empty:
        logger.info("No violations found in any file!")
        return pd.DataFrame(columns=['file', 'rule_id', 'count'])
    
    # Create summary by file and rule
    summary = all_violations.groupby(['file', 'rule_id']).size().reset_index(name='count')
    summary = summary.sort_values(['file', 'rule_id'])
    
    # Add percentage column
    file_totals = all_violations.groupby('file').size()
    summary['percentage'] = summary.apply(
        lambda row: round(row['count'] / file_totals[row['file']] * 100, 2),
        axis=1
    )
    
    # Create detailed violation details
    detailed_violations = all_violations.copy()
    detailed_violations = detailed_violations.sort_values(['file', 'rule_id', 'row_index'])
    
    if output_path:
        # Save report to CSV
        summary.to_csv(output_path / 'validation_summary.csv', index=False)
        detailed_violations.to_csv(output_path / 'validation_details.csv', index=False)
        logger.info(f"Validation report saved to {output_path}")
    
    return summary

In [7]:
def run_validation_from_files(
    raw_dir: Path,
    warehouses: pd.DataFrame,
    master_sku: Optional[Set[str]] = None,
    allowed_status: Optional[Set[str]] = None
) -> Dict[str, pd.DataFrame]:
    """
    Convenience function to validate all raw files
    
    Args:
        raw_dir: Directory containing raw CSV files
        warehouses: Warehouse master dataframe
        master_sku: Set of valid SKU codes (auto-generated if None)
        allowed_status: Set of allowed order statuses (auto-generated if None)
    
    Returns:
        Dictionary of violation dataframes
    """
    from extract import load_raw_files
    
    # Load all raw files
    raw_data = load_raw_files(raw_dir)
    
    # Validate
    violations = validate_all_files(
        raw_data=raw_data,
        warehouses=warehouses,
        master_sku=master_sku,
        allowed_status=allowed_status
    )
    
    return violations

In [8]:
"""
BASE_DIR  = Path.cwd().parent
RAW_DIR   = BASE_DIR / "data" / "raw"
sales = pd.read_csv(RAW_DIR / "sales.csv")
orders = pd.read_csv(RAW_DIR / "orders.csv")
inventory = pd.read_csv(RAW_DIR / "inventory.csv")
warehouses = pd.read_csv(RAW_DIR / "warehouses.csv")

valid_wh = set(warehouses.iloc[:, 0])
master_sku = {f"SKU-{i:04d}" for i in range(1, 41)}
allowed_status = {"Pending", "Shipped", "Delivered", "Cancelled"}

viol_sales = flag_rules(
    sales, "sales",
    id_col="transaction_id",
    date_col="date",
    sku_col="sku",
    wh_col="warehouse_id",
    qty_cols=["quantity", "revenue"],
)

viol_orders = flag_rules(
    orders, "orders",
    id_col="order_id",
    date_col="date",
    sku_col="sku",
    wh_col="warehouse_id",
    qty_cols=["ordered_quantity"],
    status_col="order_status",
)

viol_inv = flag_rules(
    inventory, "inventory",
    id_col=None,
    date_col="snapshot_date",
    sku_col="sku",
    wh_col="warehouse_id",
    qty_cols=["on_hand_qty"],
)

summary = pd.concat(
    [viol_sales, viol_orders, viol_inv],
    ignore_index=True,
).groupby(["file", "rule_id"]).size().reset_index(name="count")

print("=== Q2: Business-rule violations summary ===")
summary
"""

'\nBASE_DIR  = Path.cwd().parent\nRAW_DIR   = BASE_DIR / "data" / "raw"\nsales = pd.read_csv(RAW_DIR / "sales.csv")\norders = pd.read_csv(RAW_DIR / "orders.csv")\ninventory = pd.read_csv(RAW_DIR / "inventory.csv")\nwarehouses = pd.read_csv(RAW_DIR / "warehouses.csv")\n\nvalid_wh = set(warehouses.iloc[:, 0])\nmaster_sku = {f"SKU-{i:04d}" for i in range(1, 41)}\nallowed_status = {"Pending", "Shipped", "Delivered", "Cancelled"}\n\nviol_sales = flag_rules(\n    sales, "sales",\n    id_col="transaction_id",\n    date_col="date",\n    sku_col="sku",\n    wh_col="warehouse_id",\n    qty_cols=["quantity", "revenue"],\n)\n\nviol_orders = flag_rules(\n    orders, "orders",\n    id_col="order_id",\n    date_col="date",\n    sku_col="sku",\n    wh_col="warehouse_id",\n    qty_cols=["ordered_quantity"],\n    status_col="order_status",\n)\n\nviol_inv = flag_rules(\n    inventory, "inventory",\n    id_col=None,\n    date_col="snapshot_date",\n    sku_col="sku",\n    wh_col="warehouse_id",\n    qty_co